### ❄ schema | aggs | kpis

In [0]:
dbutils.fs.ls('/mnt/silver_layer')

In [0]:
df = spark.read.format("delta").load('/mnt/silver_layer/master_df')

In [0]:
df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Define the column renaming mapping
rename_dict = {
    'Id': 'id',
    'Firstname': 'firstname',
    'Surname': 'surname',
    'Born': 'born',
    'Died': 'died',
    'born_country': 'born_country',
    'born_country_code': 'born_country_code',
    'born_city': 'born_city',
    'died_country': 'died_country',
    'died_country_code': 'died_country_code',
    'died_city': 'died_city',
    'Gender': 'gender',
    'Year': 'year',
    'Category': 'category',
    'Motivation': 'motivation',
    'organization_name': 'organization_name',
    'organization_city': 'organization_city',
    'organization_country': 'organization_country',
    'Fullname': 'name'
}

# Apply the renaming to the DataFrame
df = df.select([F.col(col).alias(rename_dict.get(col, col)) for col in df.columns])

# Show the renamed DataFrame schema
df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Step 1: Base cleaning (removing first name, surname)
df = df.drop("firstname", "surname")

# Step 2: Country Dimension
country_dimension = df.select(
    F.col("born_country").alias("country_name"),
    F.col("born_country_code").alias("country_code")
).union(
    df.select(
        F.col("died_country").alias("country_name"),
        F.col("died_country_code").alias("country_code")
    )
).union(
    df.select(
        F.col("organization_country").alias("country_name"),
        F.lit(None).cast("string").alias("country_code")
    )
).dropna(subset=["country_name"]).dropDuplicates(["country_name"])

country_dimension = country_dimension.withColumn("country_id", F.monotonically_increasing_id())

# Add country_id to df for joins
df = df.join(country_dimension.select("country_name", "country_id").withColumnRenamed("country_name", "born_country"), on="born_country", how="left") \
       .withColumnRenamed("country_id", "born_country_id")

df = df.join(country_dimension.select("country_name", "country_id").withColumnRenamed("country_name", "died_country"), on="died_country", how="left") \
       .withColumnRenamed("country_id", "died_country_id")

df = df.join(country_dimension.select("country_name", "country_id").withColumnRenamed("country_name", "organization_country"), on="organization_country", how="left") \
       .withColumnRenamed("country_id", "organization_country_id")

# Step 3: Location Dimension (uses country_id instead of country_name)
location_dimension = df.select(
    F.col("born_city").alias("city_name"),
    F.col("born_country_id").alias("country_id")
).union(
    df.select(
        F.col("died_city").alias("city_name"),
        F.col("died_country_id").alias("country_id")
    )
).union(
    df.select(
        F.col("organization_city").alias("city_name"),
        F.col("organization_country_id").alias("country_id")
    )
).dropna(subset=["city_name", "country_id"]).dropDuplicates(["city_name", "country_id"]) \
 .withColumn("location_id", F.monotonically_increasing_id())

# Step 4: Organization Dimension
organization_dimension = df.select("organization_name", "organization_city", "organization_country_id").dropna(subset=["organization_name"]).dropDuplicates(["organization_name"])

organization_dimension = organization_dimension.join(location_dimension, 
    (organization_dimension.organization_city == location_dimension.city_name) & 
    (organization_dimension.organization_country_id == location_dimension.country_id), 
    "left"
)

organization_dimension = organization_dimension.select("organization_name", "location_id").dropDuplicates(["organization_name", "location_id"]) \
    .withColumn("organization_id", F.monotonically_increasing_id())

# Step 5: Laureate Dimension
laureate_dimension = df.select("id", "year", "category").dropDuplicates(["id", "year", "category"]).withColumnRenamed("id", "laureate_id")

# Step 6: Fact Table

# Join organization_id
df = df.join(organization_dimension, on="organization_name", how="left")

# Join born_location_id
born_loc = df.select("born_city", "born_country_id").dropDuplicates(["born_city", "born_country_id"])
born_loc = born_loc.join(location_dimension, 
    (born_loc.born_city == location_dimension.city_name) & 
    (born_loc.born_country_id == location_dimension.country_id), 
    "left"
).select("born_city", "born_country_id", "location_id") \
 .withColumnRenamed("location_id", "born_location_id")

df = df.join(born_loc, on=["born_city", "born_country_id"], how="left")

# Join died_location_id
died_loc = df.select("died_city", "died_country_id").dropDuplicates(["died_city", "died_country_id"])
died_loc = died_loc.join(location_dimension, 
    (died_loc.died_city == location_dimension.city_name) & 
    (died_loc.died_country_id == location_dimension.country_id), 
    "left"
).select("died_city", "died_country_id", "location_id") \
 .withColumnRenamed("location_id", "died_location_id")

df = df.join(died_loc, on=["died_city", "died_country_id"], how="left")

# Final fact table
fact_table = df.select(
    F.col("id").alias("laureate_id"),
    "organization_id",
    "born_location_id",
    "died_location_id",
    "name",
    "gender",
    "motivation"
)

In [0]:
print('country dimension:')
country_dimension.printSchema()
print('location dimension:')
location_dimension.printSchema()
print('organization dimension:')
organization_dimension.printSchema()
print('laureate dimension:')
laureate_dimension.printSchema()
print('fact table:')
fact_table.printSchema()

### data validation

In [0]:
# Fact vs Organization
fact_table.join(organization_dimension, "organization_id", "left_anti").count()

# Fact vs Location (born)
fact_table.join(location_dimension.withColumnRenamed("location_id", "born_location_id"), "born_location_id", "left_anti").count()

# Fact vs Location (died)
fact_table.join(location_dimension.withColumnRenamed("location_id", "died_location_id"), "died_location_id", "left_anti").count()

# Fact vs Laureate
fact_table.join(laureate_dimension, "laureate_id", "left_anti").count()

In [0]:
fact_table.select("organization_id", "born_location_id", "died_location_id", "laureate_id") \
       .filter("organization_id IS NULL OR born_location_id IS NULL OR died_location_id IS NULL OR laureate_id IS NULL") \
       .count()

In [0]:
print(organization_dimension.groupBy("organization_id").count().filter("count > 1").count())
print(location_dimension.groupBy("location_id").count().filter("count > 1").count())
print(country_dimension.groupBy("country_id").count().filter("count > 1").count())
print(laureate_dimension.groupBy("laureate_id").count().filter("count > 1").count())

In [0]:
fact_table.select("organization_id").distinct().count() == organization_dimension.select("organization_id").count()  # should match

In [0]:
# saving the facts and dimensions
# dimensions
country_dimension.write.format("delta").mode("overwrite").save("/mnt/gold_layer/dimensions/country_dimension")
location_dimension.write.format("delta").mode("overwrite").save("/mnt/gold_layer/dimensions/location_dimension")
organization_dimension.write.format("delta").mode("overwrite").save("/mnt/gold_layer/dimensions/organization_dimension")
laureate_dimension.write.format("delta").mode("overwrite").save("/mnt/gold_layer/dimensions/laureate_dimension")

# fact table
fact_table.write.format("delta").mode("overwrite").save("/mnt/gold_layer/facts/fact_table")

### aggregations | kpis

In [0]:
from pyspark.sql import functions as F

agg_laureates_by_gender = fact_table.groupBy("gender").agg(
    F.count("laureate_id").alias("total_laureates")
)

agg_laureates_by_gender.display()

In [0]:
agg_laureates_by_category = laureate_dimension.groupBy("category").agg(
    F.count("laureate_id").alias("total_laureates")
)

agg_laureates_by_category.display()

In [0]:
# Nobel Prize Distribution by Year
agg_nobel_distribution_by_year = laureate_dimension.groupBy("year").agg(
    F.count("laureate_id").alias("total_laureates")
).orderBy("year")

display(agg_nobel_distribution_by_year)

In [0]:
# Top Organizations with the Most Laureates
agg_top_organizations = fact_table.join(organization_dimension, "organization_id").groupBy("organization_name").agg(
    F.count("laureate_id").alias("total_laureates")
).orderBy(F.desc("total_laureates"))

display(agg_top_organizations)

In [0]:
# Top Countries with the Most Laureates
agg_top_countries = fact_table.join(location_dimension, fact_table["born_location_id"] == location_dimension["location_id"]) \
    .join(country_dimension, location_dimension["country_id"] == country_dimension["country_id"]) \
    .groupBy("country_name") \
    .agg(F.count("laureate_id").alias("total_laureates")) \
    .orderBy(F.desc("total_laureates"))

display(agg_top_countries)

In [0]:
kpi_laureates_by_decade = fact_table \
    .join(laureate_dimension, "laureate_id", "left") \
    .withColumn("decade", (F.col("year") / 10).cast("int") * 10) \
    .groupBy("decade") \
    .agg(F.count("laureate_id").alias("total_laureates")) \
    .orderBy("decade")

display(kpi_laureates_by_decade)

In [0]:
kpi_gender_distribution_by_category = fact_table \
    .join(laureate_dimension, "laureate_id", "left") \
    .groupBy("category", "gender") \
    .agg(F.count("laureate_id").alias("total")) \
    .orderBy("category", "gender")

display(kpi_gender_distribution_by_category)

In [0]:
# Category Trend by Each Gender Over the Decade
from pyspark.sql.functions import floor

kpi_category_trend_by_gender = laureate_dimension.join(
    fact_table, "laureate_id"
).withColumn(
    "decade", (floor(laureate_dimension["year"] / 10) * 10)
).groupBy("decade", "gender", "category").count()

display(kpi_category_trend_by_gender)

In [0]:
# Gender Ratio for Each Country
kpi_gender_ratio_by_country = fact_table.join(
    location_dimension, fact_table["born_location_id"] == location_dimension["location_id"]
).join(
    country_dimension, location_dimension["country_id"] == country_dimension["country_id"]
).groupBy("country_name", "gender").count()

display(kpi_gender_ratio_by_country)

In [0]:
from pyspark.sql.functions import min, max, countDistinct
from pyspark.sql.functions import col

org_laureate_years = laureate_dimension.join(
    fact_table, "laureate_id"
).groupBy("organization_id").agg(
    min("year").alias("first_year"),
    max("year").alias("last_year"),
    countDistinct("laureate_id").alias("total_laureates")
).withColumn(
    "active_years", col("last_year") - col("first_year") + 1
).withColumn(
    "impact_score", col("total_laureates") / col("active_years")
)

kpi_organization_impact_score = org_laureate_years
display(kpi_organization_impact_score)

In [0]:
from pyspark.sql.functions import col, count, row_number
from pyspark.sql.window import Window

# Join tables to bring in country and category
kpi_country_category = fact_table \
    .join(location_dimension, fact_table["born_location_id"] == location_dimension["location_id"]) \
    .join(country_dimension, location_dimension["country_id"] == country_dimension["country_id"]) \
    .join(laureate_dimension, "laureate_id") \
    .groupBy("country_name", "category") \
    .agg(count("laureate_id").alias("laureate_count"))

# Window to rank the top category per country
window_spec = Window.partitionBy("country_name").orderBy(col("laureate_count").desc())

kpi_top_category_by_country = kpi_country_category \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .drop("rank")

display(kpi_top_category_by_country)

In [0]:
agg_laureates_by_gender.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_laureates_by_gender")
agg_laureates_by_category.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_laureates_by_category")
agg_nobel_distribution_by_year.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_nobel_distribution_by_year")
agg_top_organizations.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_top_organizations")
agg_top_countries.write.format("delta").mode("overwrite").save("/mnt/gold_layer/aggregations/agg_top_countries")

In [0]:
kpi_laureates_by_decade.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_laureates_by_decade")
kpi_gender_distribution_by_category.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_gender_distribution_by_category")
kpi_category_trend_by_gender.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_category_trend_by_gender")
kpi_gender_ratio_by_country.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_gender_ratio_by_country")
kpi_organization_impact_score.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_organization_impact_score")
kpi_top_category_by_country.write.format("delta").mode("overwrite").save("/mnt/gold_layer/kpis/kpi_top_category_by_country")